# 案例3：电力系统暂态稳定分析

## 计算原理

暂态稳定分析研究电力系统遭受大扰动（如短路故障）后，能否恢复到新的稳定运行状态。

**经典模型（二阶运动方程）：**

$$2H \frac{d^2\delta}{dt^2} = P_m - P_e - D\frac{d\delta}{dt}$$

$$P_e = \frac{E' V_\infty}{X_\Sigma} \sin\delta$$

**等面积定则（Equal Area Criterion）：**

系统稳定的条件是减速面积 ≥ 加速面积：
$$\int_{\delta_0}^{\delta_{cr}} (P_m - P_e) d\delta + \int_{\delta_{cr}}^{\delta_{max}} (P_m - P_e) d\delta \leq 0$$

**详细模型（四阶）：** 增加励磁系统和调速系统动态

**参考教材：** 李光琦《电力系统暂态分析》第三、四章

## 案例模型：单机无穷大系统

**系统参数：**
- 暂态电势 E' = 1.2 p.u.
- 无穷大母线电压 V∞ = 1.0 p.u.
- 正常运行等值电抗 X₁ = 0.5 p.u.
- 故障期间等值电抗 X₂ = 1.0 p.u.（线路首端三相短路）
- 惯性时间常数 H = 5.0 s
- 阻尼系数 D = 2.0 p.u.
- 机械功率 Pm = 0.8 p.u.
- 初始功角 δ₀ = 30°

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

from psa4teaching.stability import (
    simulate_single_machine_infinite_bus_classic,
    simulate_single_machine_infinite_bus_detailed,
)

In [ ]:
# 系统参数
E_prime = 1.2
V_inf = 1.0
X_normal = 0.5
X_fault = 1.0
H = 5.0
D = 2.0
Pm = 0.8
delta_0 = np.radians(30)

print('单机无穷大系统参数：')
print(f'  E\' = {E_prime}, V∞ = {V_inf}')
print(f'  X_normal = {X_normal}, X_fault = {X_fault}')
print(f'  H = {H}s, D = {D}')
print(f'  Pm = {Pm}, δ₀ = {np.degrees(delta_0)}°')

# 正常运行功率极限
Pmax_normal = E_prime * V_inf / X_normal
Pmax_fault = E_prime * V_inf / X_fault
print(f'\n  正常Pmax = {Pmax_normal:.3f} p.u.')
print(f'  故障Pmax = {Pmax_fault:.3f} p.u.')

## 1. 不同故障切除时间的暂态稳定仿真

In [ ]:
# 不同的故障切除时间
clearing_times = [0.10, 0.20, 0.30, 0.40, 0.50]

results = {}
for tc in clearing_times:
    result = simulate_single_machine_infinite_bus_classic(
        E_prime, V_inf, X_normal, H, D, Pm, delta_0,
        fault_time=0.0, fault_clearing_time=tc,
        X_total_fault=X_fault,
        t_end=5.0, dt=0.005, method='rk4',
        stability_limit=180
    )
    results[tc] = result
    status = '稳定 ✓' if result.stable else '失稳 ✗'
    print(f'故障切除时间 {tc:.2f}s: {status}, 最大功角 {result.max_delta:.1f}°')

In [ ]:
# 功角曲线对比
plt.figure(figsize=(12, 8))

colors = ['#2ecc71', '#27ae60', '#f39c12', '#e74c3c', '#c0392b']
for i, (tc, result) in enumerate(results.items()):
    style = '-' if result.stable else '--'
    plt.plot(result.time, result.delta, style, linewidth=2,
             color=colors[i], label=f'tc={tc:.2f}s ({"稳定" if result.stable else "失稳"})')

plt.axhline(y=180, color='red', linestyle=':', alpha=0.5, label='180° (稳定性边界)')
plt.axvline(x=0.0, color='gray', linestyle='--', alpha=0.3)
plt.xlabel('时间 (s)', fontsize=12)
plt.ylabel('功角 δ (°)', fontsize=12)
plt.title('不同故障切除时间下的功角响应', fontsize=14)
plt.legend(loc='best', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('stability_clearing_time.png', dpi=150)
plt.show()

In [ ]:
# 电磁功率曲线
plt.figure(figsize=(10, 6))

delta_range = np.linspace(0, np.pi, 200)
Pe_normal = Pmax_normal * np.sin(delta_range)
Pe_fault = Pmax_fault * np.sin(delta_range)

plt.plot(np.degrees(delta_range), Pe_normal, 'b-', linewidth=2, label=f'正常运行 (X={X_normal})')
plt.plot(np.degrees(delta_range), Pe_fault, 'r--', linewidth=2, label=f'故障期间 (X={X_fault})')
plt.axhline(y=Pm, color='green', linestyle=':', linewidth=1.5, label=f'机械功率 Pm={Pm}')

# 标注初始功角
plt.plot(np.degrees(delta_0), Pmax_normal * np.sin(delta_0), 'ko', markersize=10)
plt.annotate(f'初始点 δ₀={np.degrees(delta_0):.0f}°', 
             xy=(np.degrees(delta_0), Pmax_normal * np.sin(delta_0)),
             xytext=(np.degrees(delta_0)+10, Pmax_normal * np.sin(delta_0)+0.15),
             arrowprops=dict(arrowstyle='->'), fontsize=10)

plt.xlabel('功角 δ (°)', fontsize=12)
plt.ylabel('电磁功率 Pe (p.u.)', fontsize=12)
plt.title('功角特性曲线（等面积定则图示）', fontsize=14)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('stability_power_angle.png', dpi=150)
plt.show()

## 2. 经典模型 vs 详细模型对比

In [ ]:
# 经典模型
result_classic = simulate_single_machine_infinite_bus_classic(
    E_prime=1.2, V_infinity=1.0, X_total=0.5,
    H=5.0, D=2.0, Pm=0.8,
    delta_0=np.radians(30),
    fault_time=0.0, fault_clearing_time=0.15,
    X_total_fault=1.0,
    t_end=5.0, dt=0.005
)

# 详细模型
result_detail = simulate_single_machine_infinite_bus_detailed(
    E_prime_0=1.2, V_infinity=1.0, X_total=0.5,
    Xd=1.8, Xd_prime=0.3, Xq=1.7,
    Td0_prime=8.0, H=5.0, D=2.0,
    Pm_0=0.8, delta_0=np.radians(30), Efd_0=2.0,
    Ka=50.0, Ta=0.05, Te=0.3,
    fault_time=0.0, fault_clearing_time=0.15,
    X_total_fault=1.0,
    t_end=5.0, dt=0.005
)

print(f'经典模型: {"稳定" if result_classic.stable else "不稳定"}，最大功角={result_classic.max_delta:.1f}°')
print(f'详细模型: {"稳定" if result_detail.stable else "不稳定"}，最大功角={result_detail.max_delta:.1f}°')

In [ ]:
# 模型对比图
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 功角对比
axes[0, 0].plot(result_classic.time, result_classic.delta, 'b-', linewidth=1.5, label='经典模型')
axes[0, 0].plot(result_detail.time, result_detail.delta, 'r--', linewidth=1.5, label='详细模型')
axes[0, 0].axhline(y=180, color='gray', linestyle=':', alpha=0.5)
axes[0, 0].set_xlabel('时间 (s)')
axes[0, 0].set_ylabel('功角 δ (°)')
axes[0, 0].set_title('功角响应对比')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 电磁功率对比
axes[0, 1].plot(result_classic.time, result_classic.Pe, 'b-', linewidth=1.5, label='经典模型')
axes[0, 1].plot(result_detail.time, result_detail.Pe, 'r--', linewidth=1.5, label='详细模型')
axes[0, 1].axhline(y=Pm, color='green', linestyle=':', label=f'Pm={Pm}')
axes[0, 1].set_xlabel('时间 (s)')
axes[0, 1].set_ylabel('电磁功率 Pe (p.u.)')
axes[0, 1].set_title('电磁功率响应对比')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 详细模型 - 励磁电压
axes[1, 0].plot(result_detail.time, result_detail.Efd, 'g-', linewidth=1.5)
axes[1, 0].set_xlabel('时间 (s)')
axes[1, 0].set_ylabel('励磁电压 Efd (p.u.)')
axes[1, 0].set_title('详细模型 - 励磁系统响应')
axes[1, 0].grid(True, alpha=0.3)

# 详细模型 - 暂态电势
axes[1, 1].plot(result_detail.time, result_detail.Eq_prime, 'm-', linewidth=1.5)
axes[1, 1].set_xlabel('时间 (s)')
axes[1, 1].set_ylabel('暂态电势 Eq\' (p.u.)')
axes[1, 1].set_title('详细模型 - 暂态电势变化')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('stability_model_comparison.png', dpi=150)
plt.show()

In [ ]:
# 阻尼系数影响
D_values = [0.0, 0.5, 2.0, 5.0]
plt.figure(figsize=(10, 6))

for D_val in D_values:
    result = simulate_single_machine_infinite_bus_classic(
        E_prime=1.2, V_infinity=1.0, X_total=0.5,
        H=5.0, D=D_val, Pm=0.8,
        delta_0=np.radians(30),
        fault_time=0.0, fault_clearing_time=0.15,
        X_total_fault=1.0,
        t_end=5.0, dt=0.005
    )
    status = '稳定' if result.stable else '失稳'
    plt.plot(result.time, result.delta, linewidth=2, label=f'D={D_val} ({status})')

plt.axhline(y=180, color='red', linestyle=':', alpha=0.5)
plt.xlabel('时间 (s)', fontsize=12)
plt.ylabel('功角 δ (°)', fontsize=12)
plt.title('不同阻尼系数下的暂态稳定响应', fontsize=14)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('stability_damping.png', dpi=150)
plt.show()

## 结果分析

1. **临界切除时间（CCT）：** 存在一个最大允许的故障切除时间，超过此时间系统失稳

2. **等面积定则：** 加速面积（故障期间Pm>Pe的区域）必须小于减速面积（故障后Pe>Pm的区域）

3. **经典模型 vs 详细模型：**
   - 经典模型假设E'恒定，计算简单但精度较低
   - 详细模型考虑励磁系统动态，更接近实际
   - 详细模型中励磁系统可提供额外阻尼，有利于稳定

4. **阻尼系数影响：** D越大振荡衰减越快，系统越容易保持稳定

5. **数值方法：** 四阶龙格-库塔法（RK4）精度高于改进欧拉法